In [1]:
import cv2
import pickle
import numpy as np
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import tensorflow as tf

MODEL_PATH    = "model_mobilenet.keras"
CLASS_IDX     = "class_indices.pkl"
IMG_SIZE      = 224

model         = tf.keras.models.load_model(MODEL_PATH)
class_indices = pickle.load(open(CLASS_IDX, "rb"))
idx_to_class  = {v: k for k, v in class_indices.items()}

correct_total = 0
tested_total  = 0

root = tk.Tk()
root.title("MobileNetV2 Tester")
root.geometry("400x500")

btn = tk.Button(root, text="Browse Image", font=("Helvetica", 14),
                bg="blue", fg="white", padx=20, pady=10)
btn.pack(pady=20)

img_label    = tk.Label(root)
img_label.pack()

result_label = tk.Label(root, text="", font=("Helvetica", 13))
result_label.pack(pady=10)

def browse():
    global correct_total, tested_total

    path = filedialog.askopenfilename(
        initialdir="data/grayscale",
        filetypes=[("JPEG", "*.jpg"), ("All", "*.*")]
    )
    if not path:
        return

    true_label = path.split("/")[-2]

    # MobileNetV2 pipeline
    img     = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    inp     = np.expand_dims(resized / 255.0, axis=0)

    preds      = model.predict(inp, verbose=0)[0]
    pred_idx   = np.argmax(preds)
    prediction = idx_to_class[pred_idx]
    confidence = preds[pred_idx] * 100

    correct = prediction == true_label
    if correct:
        correct_total += 1
    tested_total += 1

    color = "green" if correct else "red"
    result_label.config(
        text=f"True: {true_label}\nPred: {prediction}  ({confidence:.1f}%)\n{'✓ CORRECT' if correct else '✗ WRONG'}",
        fg=color
    )

    print("─" * 40)
    print(f"  Image      : {path.split('/')[-1]}")
    print(f"  True Label : {true_label}")
    print(f"  Predicted  : {prediction}")
    print(f"  Confidence : {confidence:.1f}%")
    print(f"  Result     : {'✓ CORRECT' if correct else '✗ WRONG'}")
    print(f"  Session    : {correct_total}/{tested_total} correct ({correct_total/tested_total*100:.1f}%)")
    print("─" * 40)

    # Show grayscale preview
    gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    pil_img = Image.fromarray(gray).resize((250, 250))
    tk_img  = ImageTk.PhotoImage(pil_img)
    img_label.config(image=tk_img)
    img_label.image = tk_img

btn.config(command=browse)
root.mainloop()

print(f"\n{'='*40}")
print(f"  FINAL SESSION RESULTS")
print(f"  Tested  : {tested_total} images")
print(f"  Correct : {correct_total}")
print(f"  Accuracy: {correct_total/tested_total*100:.1f}%" if tested_total > 0 else "  No images tested")
print(f"{'='*40}")

2026-05-14 22:16:47.755438: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-14 22:16:47.820008: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-14 22:16:47.871371: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-14 22:16:47.942359: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-14 22:16:47.958845: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-14 22:16:48.064144: I tensorflow/core/platform/cpu_feature_gu

────────────────────────────────────────
  Image      : 19.jpg
  True Label : F
  Predicted  : F
  Confidence : 100.0%
  Result     : ✓ CORRECT
  Session    : 1/1 correct (100.0%)
────────────────────────────────────────
────────────────────────────────────────
  Image      : 6.jpg
  True Label : 6
  Predicted  : 6
  Confidence : 100.0%
  Result     : ✓ CORRECT
  Session    : 2/2 correct (100.0%)
────────────────────────────────────────
────────────────────────────────────────
  Image      : 105.jpg
  True Label : A
  Predicted  : A
  Confidence : 100.0%
  Result     : ✓ CORRECT
  Session    : 3/3 correct (100.0%)
────────────────────────────────────────

  FINAL SESSION RESULTS
  Tested  : 3 images
  Correct : 3
  Accuracy: 100.0%
